In [54]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

In [55]:
import os
import getpass
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter groq api key")

In [56]:
"""
Multi-agent supervisor for coordinating research across multiple specialized agents.

This module implements a supervisor pattern where:
1. A supervisor agent coordinates research activities and delegates tasks
2. Multiple researcher agents work on specific sub-topics independently
3. Results are aggregated and compressed for final reporting

The supervisor uses parallel research execution to improve efficiency while
maintaining isolated context windows for each research topic.
"""
import asyncio

from typing_extensions import Literal

from langchain.chat_models import init_chat_model

from langchain_core.messages import (
    HumanMessage,
    BaseMessage,
    SystemMessage,
    ToolMessage,
    filter_messages
)

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

from langchain_groq import ChatGroq

from deepresearch.prompt import lead_researcher_prompt
from deepresearch.agents.research_agent import researcher_agent
from deepresearch.state import (
    SupervisorState,
    ConductResearch,
    ResearchComplete
)

from deepresearch.utils import get_today_str, think_tool

In [57]:
# Ensure async compatibility for Jupyter environments

try:
    import nest_asyncio

    # Only apply if running in Jupyter/IPython environment
    try:
        from IPython import get_ipython

        if get_ipython() is not None:
            nest_asyncio.apply()

    except ImportError:
        pass  # Not in Jupyter, no need for nest_asyncio

except ImportError:
    pass  # nest_asyncio not available, proceed without it

In [58]:
def get_notes_from_tool_calls(messages: list[BaseMessage]) -> list[str]:
    """Extract research notes from ToolMessage objects in supervisor message history.

    This function retrieves the compressed research findings that sub-agents
    return as ToolMessage content. When the supervisor delegates research to
    sub-agents via ConductResearch tool calls, each sub-agent returns its
    compressed findings as the content of a ToolMessage. This function
    extracts all such ToolMessage content to compile the final research notes.

    Args:
        messages: List of messages from supervisor's conversation history

    Returns:
        List of research note strings extracted from ToolMessage objects
    """

    return [
        tool_msg.content
        for tool_msg in filter_messages(messages, include_types="tool")
    ]

In [59]:
supervisor_tools = [ConductResearch, ResearchComplete,think_tool]

supervisor_model = ChatGroq(
    
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=2000
)

supervisor_model_with_tools = supervisor_model.bind_tools(supervisor_tools)

In [60]:
# System constants

# Maximum number of tool call iterations for individual researcher agents
# This prevents infinite loops and controls research depth per topic
max_researcher_iterations = 6  # Calls to think_tool + ConductResearch

# Maximum number of concurrent research agents the supervisor can launch
# This is passed to the lead_researcher_prompt to limit parallel research tasks
max_concurrent_researchers = 3

In [61]:
from deepresearch import state
async def supervisor(state: SupervisorState) -> Command[Literal["supervisor_tools"]]:
    """Coordinate research activities.

    Analyzes the research brief and current progress to decide:
    - What research topics need investigation
    - Whether to conduct parallel research
    - When research is complete

    Args:
        state: Current supervisor state with messages and research progress

    Returns:
        Command to proceed to supervisor_tools node with updated state
    """

    supervisor_messages = state.get("supervisor_messages", [])

    # Prepare system message with current date and constraints
    system_message = lead_researcher_prompt.format(
        date=get_today_str(),
        max_concurrent_research_units=max_concurrent_researchers,
        max_researcher_iterations=max_researcher_iterations
    )

    messages = [SystemMessage(content=system_message)] + supervisor_messages
    for i, msg in enumerate(messages):
        print(
            f"Message {i}: "
            f"{len(str(msg.content))} chars"
        )
    
    print("=" * 50)
    print("Supervisor iteration:",
    state.get("research_iterations", 0))

    print("Message count:",
    len(supervisor_messages))

    for i, msg in enumerate(supervisor_messages):
        print(
            f"{i}: {type(msg).__name__} "
            f"{len(str(msg.content))}"
        )
    # Make decision about next research steps
    response = await supervisor_model_with_tools.ainvoke(messages)

    return Command(
        goto="supervisor_tools",
        update={
            "supervisor_messages": [response],
            "research_iterations": state.get("research_iterations", 0) + 1
        }
    )

In [62]:
async def supervisor_tools(state: SupervisorState) -> Command[Literal["supervisor", "__end__"]]:
    """Execute supervisor decisions - either conduct research or end the process.

    Handles:
    - Executing think_tool calls for strategic reflection
    - Launching parallel research agents for different topics
    - Aggregating research results
    - Determining when research is complete

    Args:
        state: Current supervisor state with messages and iteration count

    Returns:
        Command to continue supervision, end process, or handle errors
    """

    supervisor_messages = state.get("supervisor_messages", [])
    research_iterations = state.get("research_iterations", 0)
    most_recent_message = supervisor_messages[-1]

    # Initialize variables for single return pattern
    tool_messages = []
    all_raw_notes = []
    next_step = "supervisor"  # Default next step
    should_end = False

    # Check exit criteria first
    exceeded_iterations = research_iterations >= max_researcher_iterations#returns true or false
    no_tool_calls = not most_recent_message.tool_calls#returns true or false

    research_complete = any(
        tool_call["name"] == "ResearchComplete"
        for tool_call in most_recent_message.tool_calls
    )#returns true or false

    if exceeded_iterations or no_tool_calls or research_complete:
        should_end = True
        next_step = END

    else:
        # Execute ALL tool calls before deciding next step
        try:

            # Separate think_tool calls from ConductResearch calls
            think_tool_calls = [
                tool_call
                for tool_call in most_recent_message.tool_calls
                if tool_call["name"] == "think_tool"
            ]

            conduct_research_calls = [
                tool_call
                for tool_call in most_recent_message.tool_calls
                if tool_call["name"] == "ConductResearch"
            ]

            # Handle think_tool calls (synchronous)
            for tool_call in think_tool_calls:
                observation = think_tool.invoke(tool_call["args"])

                tool_messages.append(
                    ToolMessage(
                        content=observation,
                        name=tool_call["name"],
                        tool_call_id=tool_call["id"]
                    )
                )

            # Handle ConductResearch calls (asynchronous)
            if conduct_research_calls:

                # Launch parallel research agents
                coros = [
                    researcher_agent.ainvoke(
                        {
                            "researcher_messages": [
                                HumanMessage(
                                    content=tool_call["args"]["research_topic"]
                                )
                            ],
                            "research_topic": tool_call["args"]["research_topic"]
                        }
                    )
                    for tool_call in conduct_research_calls
                ]

                # Wait for all research to complete
                tool_results = await asyncio.gather(*coros)

                # Format research results as tool messages
                # Each sub-agent returns compressed research findings in
                # result["compressed_research"]
                # We write this compressed research as the content of a
                # ToolMessage, which allows the supervisor to later retrieve
                # these findings via get_notes_from_tool_calls()

                research_tool_messages = [
                    ToolMessage(
                        content=result.get(
                            "compressed_research",
                            "Error synthesizing research report"
                        ),
                        name=tool_call["name"],
                        tool_call_id=tool_call["id"]
                    )
                    for result, tool_call in zip(
                        tool_results,
                        conduct_research_calls
                    )
                ]

                tool_messages.extend(research_tool_messages)

                # Aggregate raw notes from all research
                all_raw_notes = [
                    "\n".join(result.get("raw_notes", []))
                    for result in tool_results
                ]

        except Exception as e:
            print(f"Error in supervisor tools: {e}")
            should_end = True
            next_step = END

    # Single return point with appropriate state updates
    if should_end:
        return Command(
            goto=next_step,
            update={
                "notes": get_notes_from_tool_calls(supervisor_messages),
                "research_brief": state.get("research_brief", "")
            }
        )

    else:
        return Command(
            goto=next_step,
            update={
                "supervisor_messages": tool_messages,
                "raw_notes": all_raw_notes
            }
        )

In [63]:
supervisor_builder = StateGraph(SupervisorState)

supervisor_builder.add_node(
    "supervisor",
    supervisor
)

supervisor_builder.add_node(
    "supervisor_tools",
    supervisor_tools
)

supervisor_builder.add_edge(
    START,
    "supervisor"
)

supervisor_agent = supervisor_builder.compile()

In [64]:
from utils import format_messages
from langchain_core.messages import HumanMessage

research_brief = """
I want to determine the best Indian Premier League (IPL) team by analyzing two primary metrics: ...

Please research all IPL teams, including defunct ones, from the league's inception in 2008 to the most recently completed season.

The final output should be a comparative analysis that ranks or lists all teams based on both their total number of IPL ...

Please use official sources like the IPL's official website (iplt20.com) or highly reputable sports statistics sites ...
"""

result = await supervisor_agent.ainvoke(
    {
        "supervisor_messages": [
            HumanMessage(
                content=f"{research_brief}."
            )
        ]
    }
)

format_messages(
    result["supervisor_messages"]
)

Message 0: 3225 chars
Message 1: 480 chars
Supervisor iteration: 0
Message count: 1
0: HumanMessage 480

=== RAW TAVILY RESULTS ===
[{'query': 'list of all Indian Premier League teams, wins, and win-loss ratio, official sources, inception 2008 to latest season', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://chrysalishigh.com/blog/uncategorized/indian-premier-league-history', 'title': 'Indian Premier League History & IPL Winners List (2008–2026)', 'content': 'Explore the Indian Premier League cricket history, IPL winners list, most successful teams in IPL, and latest IPL cricket schedule till 2026.', 'score': 0.99979657, 'raw_content': '![](data:image/svg+xml,%3Csvg%20xmlns=\'http://www.w3.org/2000/svg\'%20viewBox=\'0%200%201990%20762\'%3E%3C/svg%3E)\n![](https://chrysalishigh.com/wp-content/uploads/2022/06/CH-School-14-years-logo.png)\n\n# History of IPL: From 2008 to IPL 2026 – Growth, Milestones and Legendary Moments\n\nThe Indian Premier Lea

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ I want to determine the best Indian Premier League (IPL) team by analyzing two primary metrics: ...             │
│                                                                                                                 │
│ Please research all IPL teams, including defunct ones, from the league's inception in 2008 to the most recently │
│ completed season.                                                                                               │
│                                                                                                                 │
│ The final output should be a comparative analysis that ranks or lists all teams based on both their total       │
│ number of IPL ...                                                                                               │
│                                                                                                                 │
│ Please use official sources like the IPL's official website (iplt20.com) or highly reputable sports statistics  │
│ sites ...                                                                                                       │
│ .                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: think_tool                                                                                        │
│    Args: {                                                                                                      │
│   "reflection": "To determine the best Indian Premier League team, we need to analyze two primary metrics.      │
│ First, we must identify all IPL teams, including defunct ones, from the league's inception in 2008 to the most  │
│ recently completed season. Then, we will research each team's total number of IPL wins and their win-loss       │
│ ratio. We will use official sources like the IPL's official website or highly reputable sports statistics sites │
│ for accurate data. Given the complexity of the task, we will start by researching all IPL teams and their       │
│ performance metrics."                                                                                           │
│ }                                                                                                               │
│    ID: htsz9vtze                                                                                                │
│                                                                                                                 │
│ 🔧 Tool Call: ConductResearch                                                                                   │
│    Args: {                                                                                                      │
│   "research_topic": "Provide a list of all Indian Premier League teams, including defunct ones, from the        │
│ league's inception in 2008 to the most recently completed season, along with their total number of IPL wins and │
│ win-loss ratio, using official sources like the IPL's official website or highly reputable sports statistics    │
│ sites."                                                                                                         │
│ }                                                                                                               │
│    ID: v2021ympz                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯